# 3. Variables que no usan el target

Salen de lo que se vio en la exploración. El código está en la sección de features de `datos.py` y hay tres tipos.

1. Del propio item, como stock, precio, fechas y fotos.
2. Palabras clave en el título y la garantía.
3. Perfiles, que describen cómo publica cada vendedor, cada categoría y los títulos con las mismas palabras. Se calculan sobre train y test juntos.

In [1]:
import sys
import warnings

sys.path.insert(0, "..")  # Los módulos están en la raíz del proyecto
warnings.filterwarnings("ignore")
%matplotlib inline

import pandas as pd

import datos
import diccionario
import modelo
import pruebas

crudo_train, crudo_test = datos.cargar_train_test()
train, test = datos.construir(crudo_train, crudo_test)
y = modelo.y_binario(train)
X = modelo.preparar(train)
print("Entran", X.shape[1], "columnas al modelo, antes del target encoding")

Entran 157 columnas al modelo, antes del target encoding


## Cuántas variables hay de cada grupo

El significado de cada una está en `docs/DICCIONARIO.md`.

In [2]:
grupo = {v.nombre: v.grupo for v in diccionario.VARIABLES}
pd.Series([grupo[c] for c in X.columns]).value_counts().rename("variables en el modelo").to_frame()

,variables en el modelo
Perfil del vendedor,26
Features derivadas del item,25
Medios de pago,16
Perfil de la categoría,16
Palabras clave en título y garantía,14
Contenido de la publicación,12
Perfil por palabra del título,10
"Ficha técnica (vehículos, inmuebles y repuestos)",9
Formato de las fotos,8
Tipo de publicación y estado,7


## Ejemplos de cuánto separan

La tasa de usados cambia mucho según estas señales. El caso más raro es el desfasaje en la duración, donde unos segundos de más son mucho más comunes en usados (D25).

In [3]:
ejemplos = {
    "Stock inicial de 1": train["iq_es_1"],
    "Título dice nuevo": train["t_nuevo"],
    "Título describe estado de uso": train["t_estado_de_uso"],
    "Garantía con plazo": train["w_plazo"],
    "Desfasaje en la duración": train["duracion_resto_seg"] > 0,
    "Vendedor con un solo item": train["seller_n_items"] == 1,
}
pd.DataFrame([
    {"señal": n, "% de los items": 100 * s.mean(), "% usados con la señal": 100 * y[s].mean(),
     "% usados sin la señal": 100 * y[~s].mean()}
    for n, s in ejemplos.items()
]).round(1).set_index("señal")

,% de los items,% usados con la señal,% usados sin la señal
señal,,,
Stock inicial de 1,70.2,63.9,4.9
Título dice nuevo,4.2,12.0,47.8
Título describe estado de uso,2.2,96.8,45.2
Garantía con plazo,8.3,7.3,49.8
Desfasaje en la duración,4.4,71.7,45.1
Vendedor con un solo item,25.0,50.6,44.9


## Estas variables no miran el target

Tres de los cinco controles de leakage, que están en `pruebas.py`. Una lista vacía significa que no se encontró ningún problema. Los otros dos están en el notebook siguiente.

In [4]:
for control in [pruebas.invarianza_al_target, pruebas.mismas_columnas, pruebas.disponibilidad_en_test]:
    print(control.__doc__, "Problemas encontrados", control(crudo_train, crudo_test, train, test))

Mezclar o borrar `condition` no puede cambiar ninguna feature. Problemas encontrados []
Train y test tienen las mismas features. Problemas encontrados []
Ninguna feature tiene muchos más nulos en test que en train. Problemas encontrados []
